In [0]:
import datetime

import pyspark
import pandas as pd

from config.settings import *

from py4j.protocol import Py4JJavaError

In [0]:
# Pull GCP credentials securely from Databricks Secrets and set them as configuration
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
# Columns to keep for analysis
COLUMNS = [
    "id", "ident", "type", "name",
    "latitude_deg", "longitude_deg", "iso_country", "iso_region"
]

# Top 25 busiest US airports (2025)
AIRPORT_ID = [
    3384, 3488, 3754, 3486, 3632,
    3622, 3670, 3631, 3685, 3878,
    3457, 3875, 3772, 3521, 3604,
    3422, 3709, 3497, 3643, 3531,
    3771, 3602, 3887, 3862, 3420
]

RAW = f"gs://{gcp_project_id}/raw/airports"
CLEAN = f"gs://{gcp_project_id}/clean/airports"


In [0]:
def preprocess_raw(rdd: pyspark.RDD) -> pyspark.RDD:
    """Pre-processes/Cleans raw data using Spark"""

    header = rdd.first()
    columns = header.split(",")

    fields = [(i, col) for i, col in enumerate(columns) if col in COLUMNS]
    idx, field = zip(*fields)

    # 1. Filter out header row
    # 2. Map each line as a list of each column values
    data_rdd = rdd.filter(lambda row: row != header) \
                    .map(lambda line: line.split(","))

    # 1. Filter top 25 busiest airports (in 2025) to keep by ID
    # 2. Map only fields to keep for analysis
    # 3. Join all fields into a single string for each airport
    airport_rdd = data_rdd.filter(lambda row: int(row[0]) in AIRPORT_ID) \
                            .map(lambda field: [field[i] for i in idx]) \
                            .map(lambda field: ",".join(field))

    return airport_rdd

In [0]:
rdd = sc.textFile(RAW)
preprocessed_rdd = preprocess_raw(rdd)

preprocessed_rdd.collect()

In [0]:
try:
    preprocessed_rdd.saveAsTextFile(CLEAN)
    status = "SUCCESS"
except Py4JJavaError as e:
    status = "FAILED"
    print(f"Error saving raw data to GCS: {str(e.java_exception)=}")
    raise

print({
    "date": datetime.date.today().strftime("%Y-%m-%d"),
    "rdd_stored": CLEAN,
    "status": status
})